In [1]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.config import ANOMALY_SIGMA, DATA_PROCESSED, HEALTHY_RATIO, IF_CONTAMINATION, MODELS_DIR, OC_SVM_NU, REPORTS_DIR, SEED, TRANSITION_RATIO
from src.models import compute_metrics
from src.models.baselines import fit_isolation_forest, fit_one_class_svm, rolling_rms_scores, score_isolation_forest, score_one_class_svm
from src.visualization.plots import plot_anomaly_scores, plot_pr_curves, plot_roc_curves

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
FIGURES_DIR = REPORTS_DIR / "figures"
METRICS_DIR = MODELS_DIR / "metrics"

In [3]:
femto_features = pd.read_parquet(DATA_PROCESSED / "femto_features_all_bearings.parquet")

In [4]:
bearing_features = femto_features[femto_features["bearing_id"] == "Bearing1_1"].copy()
bearing_features = bearing_features.sort_values(["file_idx", "window_idx"]).reset_index(drop=True)

In [5]:
file_count = int(bearing_features["file_idx"].max() + 1)
healthy_end = int(file_count * HEALTHY_RATIO)
anomaly_start = int(file_count * (HEALTHY_RATIO + TRANSITION_RATIO))
label_mask = (bearing_features["file_idx"] < healthy_end) | (bearing_features["file_idx"] >= anomaly_start)
labeled_features = bearing_features.loc[label_mask].copy()
labeled_features["label"] = (labeled_features["file_idx"] >= anomaly_start).astype(int)
y_true = labeled_features["label"].to_numpy()

In [6]:
feature_columns = labeled_features.select_dtypes(include=np.number).columns.drop(["file_idx", "window_idx", "label"])
X_labeled = labeled_features[feature_columns].to_numpy()
X_train = labeled_features.loc[labeled_features["label"] == 0, feature_columns].to_numpy()

In [7]:
rms_train_ratio = int((labeled_features["label"] == 0).sum()) / len(labeled_features)
rms_scores, rms_sigma_threshold = rolling_rms_scores(
    labeled_features["ch_h_rms"].to_numpy(), rms_train_ratio, ANOMALY_SIGMA
)

In [8]:
if_clf, if_scaler = fit_isolation_forest(X_train, IF_CONTAMINATION)
if_scores = score_isolation_forest(if_clf, if_scaler, X_labeled)

In [9]:
oc_svm_clf, oc_svm_scaler = fit_one_class_svm(X_train, OC_SVM_NU)
oc_svm_scores = score_one_class_svm(oc_svm_clf, oc_svm_scaler, X_labeled)

In [10]:
score_table = {
    "Rolling RMS": rms_scores,
    "Isolation Forest": if_scores,
    "One-Class SVM": oc_svm_scores,
}
metrics_rows = []

for model_name, scores in score_table.items():
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-10)
    threshold = thresholds[np.argmax(f1_values)]
    metrics_row = compute_metrics(y_true, scores, threshold)
    metrics_rows.append({"model": model_name, "threshold": threshold, **metrics_row})

In [11]:
baseline_metrics = pd.DataFrame(metrics_rows)
baseline_metrics.loc[baseline_metrics["model"] == "Rolling RMS", "sigma_threshold"] = rms_sigma_threshold
print(baseline_metrics.to_string(index=False))

           model  threshold  roc_auc   pr_auc       f1  precision   recall  time_to_detection  sigma_threshold
     Rolling RMS   0.747281 0.999998 0.999995 0.999554   1.000000 0.999109           0.747667         0.645813
Isolation Forest   0.059699 0.993394 0.963745 0.969618   0.945008 0.995544           0.747667              NaN
   One-Class SVM  23.525424 0.999796 0.999393 0.986784   0.975610 0.998217           0.747667              NaN


In [12]:
for model_name, scores in score_table.items():
    threshold = baseline_metrics.loc[baseline_metrics["model"] == model_name, "threshold"].iloc[0]
    save_name = model_name.lower().replace(" ", "_").replace("-", "_")
    plot_anomaly_scores(
        scores, model_name, FIGURES_DIR / f"bearing1_1_{save_name}_scores.png", threshold, len(scores) - 1
    )

In [13]:
roc_results = {}

for model_name, scores in score_table.items():
    fpr_array, tpr_array, _ = roc_curve(y_true, scores)
    roc_results[model_name] = (fpr_array, tpr_array, roc_auc_score(y_true, scores))

plot_roc_curves(roc_results, FIGURES_DIR / "bearing1_1_baseline_roc_curves.png")

In [14]:
pr_results = {}

for model_name, scores in score_table.items():
    precision_array, recall_array, _ = precision_recall_curve(y_true, scores)
    pr_results[model_name] = (precision_array, recall_array, average_precision_score(y_true, scores))

plot_pr_curves(pr_results, FIGURES_DIR / "bearing1_1_baseline_pr_curves.png")

In [15]:
METRICS_DIR.mkdir(parents=True, exist_ok=True)
baseline_metrics.to_csv(METRICS_DIR / "baseline_metrics.csv", index=False)

The baseline comparison is intentionally score-based: all three methods receive the same engineered FEMTO windows and the threshold is selected only on Bearing1_1. If RMS performs close to the learned baselines, that is technically meaningful rather than disappointing, because bearing degradation often expresses first as rising vibration energy. Very high AUC should be treated carefully because the labels are time-derived and the late-life fault interval is far from the healthy interval.